

# EDA on Real, Scraped Data - Massachusetts Income by Town
### OPIM 5641 - Business Decision Modeling · Module 1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5641-notebooks/blob/main/1_EDA_Intro/M1_EDA_WikiIncome.ipynb)

*Run me top to bottom - **Runtime → Run all**. The data comes straight from Wikipedia, so there's nothing to upload.*

# EDA on Real, Scraped Data

------------------------------------------------

Before we can optimize anything, we need to understand our data - and the fastest way to learn EDA is on data you actually scraped yourself. In this notebook we grab a real table off Wikipedia (every town in Massachusetts with its income statistics), clean it up, and squeeze some insight out of it. This is the exact loop you will repeat all semester: **question → load → clean → describe → visualize → insight**.

🔷 **The nugget:** real data never arrives clean. The two "problems" we hit in this notebook - a website that blocks naive scrapers, and income values that are censored at `$250,000+` - are not annoyances, they ARE the job.

# Step 1: Scrape the table from Wikipedia
Our dataset is the [List of Massachusetts locations by per capita income](https://en.wikipedia.org/wiki/List_of_Massachusetts_locations_by_per_capita_income) - 341 towns and cities with per capita income, median household income, median family income, population and households. Rich enough to actually learn something.

Once upon a time you could just call `pd.read_html(url)` and be done. Try it - it fails now with `HTTP Error 403: Forbidden`. Wikipedia sees a request with no browser "User-Agent" and assumes you are a bot (fair!).

**Remember:** a 403 is the server saying "I see you, and no." It is not a bug in pandas.

In [ ]:
# import our libraries - every line has a comment, that's the house style!
import pandas as pd               # tables
import requests                   # fetching web pages politely
import io                         # wrapping text so read_html can parse it
import matplotlib.pyplot as plt   # plots

In [ ]:
# the page we want to scrape
url = 'https://en.wikipedia.org/wiki/List_of_Massachusetts_locations_by_per_capita_income'

# this used to work... now it throws HTTP Error 403: Forbidden. Uncomment and see for yourself!
# df = pd.read_html(url)[2]

The fix is simple and it is a real-world lesson: fetch the page yourself with a **User-Agent header** (telling the server who you are), then hand the HTML text to `pd.read_html()`.

In [ ]:
# fetch the page with a User-Agent header - now Wikipedia is happy to talk to us
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0 (OPIM 5641 teaching demo)'})
print(response.status_code)   # 200 means OK!

In [ ]:
# parse ALL the tables on the page from the HTML we already downloaded
tables = pd.read_html(io.StringIO(response.text))

# how many tables did we get? which one is ours?
for i, t in enumerate(tables):
    print(i, t.shape)

Table index 2 has 341 rows - that's our municipality table. Grabbing the right table by index is part of scraping; always look before you leap.

**On your own:** what is table index 1? Print it and figure out what it represents.

In [ ]:
# grab the 341-town municipality table
df = tables[2]
df.head()

# Step 2: Clean the scrape
Look at the income columns - they are **strings** with `$` and `,` in them, so we can't do math yet. And here is the fun one: the very richest towns (Dover, Weston...) don't report a number at all - they show **`$250,000+`**. That is called a *censored* value: we know income is at least 250K, but not how much more.

There is no single right answer for censored data. Today we make the simplest defensible choice - treat `$250,000+` as `250000` - and we say so out loud. In a report, that footnote is the difference between analysis and hand-waving.

**Caution:** `pd.to_numeric(errors='coerce')` silently turns anything unparseable into `NaN`. Powerful, but always count your `NaN`s afterward so you know what you just threw away.

In [ ]:
# check the dtypes - the income columns are 'object' (strings), not numbers
df.info()

In [ ]:
# clean the three income columns: strip $ , and + then convert to numeric
money_cols = ['Per capita income', 'Median household income', 'Median family income']

for col in money_cols:
    df[col] = (df[col]
               .astype(str)
               .str.replace('$', '', regex=False)   # drop dollar signs
               .str.replace(',', '', regex=False)   # drop thousands commas
               .str.replace('+', '', regex=False))  # the $250,000+ censored towns
    df[col] = pd.to_numeric(df[col], errors='coerce')

# how many NaNs did coercion create? (check your work!)
df[money_cols].isna().sum()

In [ ]:
# Households has thousands commas too - strip and convert (Population is already numeric)
df['Households'] = pd.to_numeric(df['Households'].astype(str).str.replace(',', '', regex=False), errors='coerce')

# confirm the dtypes are fixed
df.dtypes

# Step 3: Describe the data
Now that everything is numeric, `.describe()` gives us the five-number summary (and more) in one shot.

In [ ]:
# the classic first look
df[money_cols + ['Population']].describe()

In [ ]:
# how many towns are in each county?
df['County'].value_counts()

In [ ]:
# what municipality types do we have?
df['Type'].value_counts()

# Step 4: Visualize
A histogram for the shape of the distribution, a bar chart to compare counties, and a scatterplot to hunt for relationships. Add labels and titles - unlabeled plots don't count.

In [ ]:
# distribution of per capita income across all 341 towns
df['Per capita income'].plot(kind='hist', bins=30, color='darkorange', edgecolor='black', figsize=(8,5))
plt.title('Per Capita Income Across Massachusetts Towns')
plt.xlabel('Per capita income ($)')
plt.ylabel('Number of towns')
plt.show()

In [ ]:
# top 10 and bottom 10 towns by per capita income
print(df.nlargest(10, 'Per capita income')[['Municipality', 'County', 'Per capita income']])
print()
print(df.nsmallest(10, 'Per capita income')[['Municipality', 'County', 'Per capita income']])

In [ ]:
# average per capita income by county, sorted
df.groupby('County')['Per capita income'].mean().sort_values().plot(kind='barh', color='steelblue', figsize=(8,6))
plt.title('Mean Per Capita Income by County')
plt.xlabel('Mean per capita income ($)')
plt.show()

In [ ]:
# is income related to population? log scale on x because Boston is huge
df.plot(kind='scatter', x='Population', y='Per capita income', alpha=0.6, color='darkorange', figsize=(8,5))
plt.xscale('log')
plt.title('Per Capita Income vs. Population (log scale)')
plt.xlabel('Population (log scale)')
plt.ylabel('Per capita income ($)')
plt.show()

## Map it - counties as a choropleth
Massachusetts data begs for a map. `geopandas` (preinstalled on Colab) reads the Census Bureau's county boundary file straight from a URL, we aggregate our 341 towns up to county averages, join the numbers onto the shapes, and color the counties by each income column.

**Caution:** we are averaging *towns within a county*, so tiny Gosnold counts exactly as much as Boston. That is a fine first look, but it is NOT the county's true income - for that you would weight by population. Keep that footnote in mind before you quote these maps.

In [ ]:
# geopandas reads shapefiles straight from a URL - this is the Census Bureau's county map
import geopandas as gpd

counties = gpd.read_file('https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_county_500k.zip')

# Massachusetts is state FIPS code 25 - filter to its 14 counties
ma = counties[counties['STATEFP'] == '25']
ma.plot(edgecolor='black', color='lightgray', figsize=(8,5))
plt.title('The 14 Counties of Massachusetts')
plt.axis('off')
plt.show()

In [ ]:
# average each income column by county from our towns table
county_stats = df.groupby('County')[money_cols].mean().reset_index()

# join the numbers onto the map - the census NAME column matches our County column
ma_map = ma.merge(county_stats, left_on='NAME', right_on='County')
ma_map[['NAME'] + money_cols]

In [ ]:
# one map per income column - counties colored by the mean of their towns
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, col in zip(axes, money_cols):
    ma_map.plot(column=col, cmap='YlOrRd', legend=True, edgecolor='black', linewidth=0.5, ax=ax, legend_kwds={'shrink': 0.5})
    ax.set_title(col)
    ax.axis('off')
plt.suptitle('Massachusetts County Averages (from 341 towns)', fontsize=14)
plt.show()

Cape Cod and the islands (Nantucket, Dukes) glow on per capita income while Hampden County (Springfield) sits at the bottom - and the three income definitions do NOT tell identical stories. That contrast is exactly why you look at more than one column.

**On your own:** redo the choropleth as a **population-weighted** county average (weight each town by `Population` before averaging). Which counties move the most, and why?

## Apply to Connecticut
Same skills, our home state - with a plot twist. **Connecticut retired its counties in 2022**: the Census Bureau now publishes 9 *planning regions* instead of the 8 legacy counties. The Wikipedia table helpfully carries BOTH columns, so we'll map by planning region (that's what the 2022 shapes are).

And a second real-world lesson: the region names on Wikipedia (`Western CT`, `Capitol Region`) do NOT exactly match the census names (`Western Connecticut`, `Capitol`). Join keys almost never line up in the wild - you normalize them, then merge.

**Remember:** when a merge comes back with fewer rows than you expected, the join key is almost always the culprit. Print both sides' unique values and compare.

In [ ]:
# scrape the Connecticut page - same User-Agent trick
ct_url = 'https://en.wikipedia.org/wiki/List_of_Connecticut_locations_by_per_capita_income'
ct_response = requests.get(ct_url, headers={'User-Agent': 'Mozilla/5.0 (OPIM 5641 teaching demo)'})
ct_tables = pd.read_html(io.StringIO(ct_response.text))

# table index 3 = the 182 places, with BOTH County and Planning Region columns
ct = ct_tables[3]

# clean the money columns exactly like we did for Massachusetts
for col in money_cols:
    ct[col] = (ct[col]
               .astype(str)
               .str.replace('$', '', regex=False)
               .str.replace(',', '', regex=False)
               .str.replace('+', '', regex=False))
    ct[col] = pd.to_numeric(ct[col], errors='coerce')

ct.head()

In [ ]:
# the join-key problem in the flesh: Wikipedia vs census spellings
print(sorted(ct['Planning Region'].dropna().unique()))
print(sorted(counties[counties['STATEFP'] == '09']['NAME'].unique()))

In [ ]:
# normalize the Wikipedia names to match the census: CT -> Connecticut, drop ' Region'
ct['Planning Region'] = (ct['Planning Region']
                         .str.replace('CT', 'Connecticut', regex=False)
                         .str.replace(' Region', '', regex=False))

# average each income column by planning region
ct_stats = ct.groupby('Planning Region')[money_cols].mean().reset_index()

# Connecticut is state FIPS 09 - same shapefile we already downloaded
ct_shapes = counties[counties['STATEFP'] == '09']
ct_map = ct_shapes.merge(ct_stats, left_on='NAME', right_on='Planning Region')
print(len(ct_map), 'of 9 regions matched')   # check your work - all 9 or go debug the keys!
ct_map[['NAME'] + money_cols]

In [ ]:
# one map per income column - planning regions colored by the mean of their places
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, col in zip(axes, money_cols):
    ct_map.plot(column=col, cmap='YlOrRd', legend=True, edgecolor='black', linewidth=0.5, ax=ax, legend_kwds={'shrink': 0.5})
    ax.set_title(col)
    ax.axis('off')
plt.suptitle('Connecticut Planning Region Averages (from 182 places)', fontsize=14)
plt.show()

Western Connecticut (Greenwich, Darien, New Canaan) glows just like Cape Cod did in Massachusetts, and the Naugatuck Valley / Northeastern corner tell the other side of the story.

**On your own:**
1. The table also has the **legacy `County` column** - redo the choropleth by county using an older boundary file (`cb_2020_us_county_500k.zip` still has the 8 CT counties). Which view do you find more informative?
2. Which planning region has the widest income *gap* between its richest and poorest place?
3. Stack MA and CT into one DataFrame (add a `State` column) and compare their per capita income distributions with two overlaid histograms.

# Step 5: From plot to insight
An insight is one sentence, backed by a number, that a smart non-technical person would care about. Examples of the *form* (check the numbers yourself before quoting them!):

- "Nantucket and Dukes counties have few towns, so their county means swing on a handful of observations."
- "The income distribution is right-skewed - a long tail of wealthy towns pulls the mean above the median."

An **over-claim** looks like: "bigger towns cause lower incomes." Our scatterplot shows association, not causation - we never get to say "cause" from EDA alone.

**On your own:**
1. State ONE insight of your own in a single sentence, backed by a specific number from this notebook.
2. Recreate the histogram for `Median family income`. Does the $250,000+ censoring distort it more or less than per capita income? Why?
3. Which county has the biggest *gap* between its richest and poorest town? (Hint: `groupby` + `max` - `min`.)

------------------------------------------------

**Bottom line:** scrape politely (User-Agent header), clean deliberately (say what you did with `$250,000+`), describe before you visualize, and end with an insight a decision-maker could act on. Next up: when you have NO data at all... you simulate it. See you in Monte Carlo.